In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.svm import SVR
from sklearn.neighbors import NearestNeighbors
from concurrent.futures import ProcessPoolExecutor

In [4]:
# Load the dataset
file_path = r'D:\Manoj_Honors\Merged_Data.csv'
df = pd.read_csv(file_path, parse_dates=['Time'])

# Number of nearest neighbors
K = 5

# Prepare unique coordinates and Station_IDs
unique_stations = df[['Station_ID', 'Lat', 'Lon']].drop_duplicates().reset_index(drop=True)
coords = unique_stations[['Lat', 'Lon']].values

# Initialize NearestNeighbors model and fit to coordinates
nn = NearestNeighbors(n_neighbors=K + 1, algorithm='ball_tree')
nn.fit(coords)

# Find K nearest neighbors for each station
distances, indices = nn.kneighbors(coords)

# Create a dictionary mapping each Station_ID to its K nearest neighbors
station_neighbors = {
    unique_stations.iloc[i]['Station_ID']: unique_stations.iloc[indices[i][1:]]['Station_ID'].values
    for i in range(len(unique_stations))
}

# Prepare an empty DataFrame to store spatial averages
spatial_averages_df = pd.DataFrame()

# Calculate spatial averages by iterating through each timestamp
for time, time_df in df.groupby('Time'):
    # Join the current timestamp data with neighboring data for each station
    time_spatial_df = time_df.copy()
    for pollutant in ['PM2.5', 'Ozone', 'NO2']:
        spatial_avg = []
        for idx, row in time_spatial_df.iterrows():
            station_id = row['Station_ID']
            neighbors = station_neighbors.get(station_id, [])
            # Calculate spatial average for current station's neighbors
            neighbors_data = time_df[time_df['Station_ID'].isin(neighbors)]
            avg = neighbors_data[pollutant].mean() if not neighbors_data.empty else np.nan
            spatial_avg.append(avg)
        # Store the results in the DataFrame
        time_spatial_df[f'Spatial_Avg_{pollutant}'] = spatial_avg
    spatial_averages_df = pd.concat([spatial_averages_df, time_spatial_df])

# Save the updated DataFrame to a new file

file_path_with_spatial = r'D:\Manoj_Honors\Merged_Data(Spatio-temporal).csv'
spatial_averages_df.to_csv(file_path_with_spatial, index=False)




In [7]:
all_available = pd.read_csv(r'C:\Users\Faculty\Desktop\Manoj_Honors\RF_csv\all_available(t-1)_with_spatial.csv')
all_available.columns

Index(['Unnamed: 0', 'Time', 'Station_ID', 'PM2.5', 'Ozone', 'NO2', 'Date',
       'Average_PM2.5_t-1', 'Average_Ozone_t-1', 'Average_NO2_t-1', 'Station',
       'Lat', 'Lon', 'Spatial_Avg_PM2.5', 'Spatial_Avg_Ozone',
       'Spatial_Avg_NO2'],
      dtype='object')

RFR

In [2]:
FT = pd.read_csv(r'C:\Users\Faculty\Desktop\Manoj_Honors\RF_csv\FT.csv')
FT.head()

,Unnamed: 0,Time,Station_ID,PM2.5,Ozone,NO2,Date,Average_PM2.5_t-1,Average_Ozone_t-1,Average_NO2_t-1,Station,Lat,Lon,Spatial_Avg_PM2.5,Spatial_Avg_Ozone,Spatial_Avg_NO2,PM2.5_FT,Ozone_FT,NO2_FT
0,0,2023-01-01 00:00:00,0,134.00,3.3,30.90,2023-01-01,129.682927,35.512195,23.029268,Alipur Delhi - DPCC,28.815329,77.153010,188.000000,12.600000,31.325000,118.098285,29.592102,35.150609
1,1,2023-01-01 00:00:00,1,148.00,8.8,86.10,2023-01-01,162.608696,13.608696,94.628261,Anand Vihar Delhi - DPCC,28.646835,77.316032,169.075000,4.402500,39.767500,118.099616,29.591767,35.150695
2,2,2023-01-01 00:00:00,2,147.00,2.4,39.00,2023-01-01,166.311111,10.951111,37.033333,Ashok Vihar Delhi - DPCC,28.695381,77.181665,139.533333,13.756667,36.816667,118.100946,29.591432,35.150781
3,3,2023-01-01 00:00:00,4,185.00,8.2,13.50,2023-01-01,175.600000,15.164444,15.466667,Bawana Delhi - DPCC,28.776200,77.051074,184.400000,11.100000,27.800000,118.102277,29.591097,35.150867
4,4,2023-01-01 00:00:00,6,262.07,17.9,26.95,2023-01-01,163.422000,26.926667,26.674222,CRRI Mathura Road Delhi - IMD,28.551201,77.273574,179.800000,4.460000,45.368000,118.103608,29.590762,35.150953


In [ ]:
# Load the dataset
all_available = pd.read_csv(r'C:\Users\Faculty\Desktop\Manoj_Honors\RF_csv\FT.csv')


# Prepare the feature set and target variable
X = all_available[['Time', 'Lat','Lon','PM2.5','NO2']]
y = all_available['Ozone']

# Convert 'Time' to numeric if it's in datetime format
X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds

# Initialize the Random Forest model
rf = RandomForestRegressor(n_estimators=100, random_state=12)

# Initialize KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=12)

# Prepare lists to store scores for each fold
mse_scores = []
mae_scores = []
r2_scores = []
mape_scores = []
nrmse_scores = []

# Perform cross-validation
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Fit the model on the training data
    rf.fit(X_train, y_train)

    # Make predictions on the test data
    y_pred = rf.predict(X_test)

    # Calculate evaluation metrics
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    nrmse = np.sqrt(mse) / (y_test.max() - y_test.min())

    mse_scores.append(mse)
    mae_scores.append(mae)
    r2_scores.append(r2)
    mape_scores.append(mape)
    nrmse_scores.append(nrmse)

# Calculate mean and standard deviation for each metric
mean_mse = np.mean(mse_scores)
std_mse = np.std(mse_scores)
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)
mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)
mean_mape = np.mean(mape_scores)
std_mape = np.std(mape_scores)
mean_nrmse = np.mean(nrmse_scores)
std_nrmse = np.std(nrmse_scores)

# Print evaluation metrics
print(f"Mean Squared Error: {round(mean_mse, 2)} ± {round(std_mse, 2)}")
print(f"Mean Absolute Error: {round(mean_mae, 2)} ± {round(std_mae, 2)}")
print(f"R^2 Score: {round(mean_r2, 2)} ± {round(std_r2, 2)}")
print(f"Mean Absolute Percentage Error: {round(mean_mape, 2)}% ± {round(std_mape, 2)}%")
print(f"Normalized RMSE: {round(mean_nrmse, 4)} ± {round(std_nrmse, 4)}")


C:\Users\Faculty\AppData\Local\Temp\ipykernel_35448\1373657655.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds


Mean Squared Error: 227.79 ± 2.85
Mean Absolute Error: 8.43 ± 0.03
R^2 Score: 0.8 ± 0.0
Mean Absolute Percentage Error: 101.12% ± 1.68%
Normalized RMSE: 0.0755 ± 0.0005


XG-Boost

In [4]:
# Load the dataset
all_available = pd.read_csv(r'C:\Users\Faculty\Desktop\Manoj_Honors\RF_csv\all_available(t-1)_with_spatial.csv')

# Prepare the feature set and target variable
X = all_available[['Time', 'Lat', 'Lon', 'NO2', 'PM2.5','Spatial_Avg_NO2', 'Spatial_Avg_PM2.5']]
y = all_available['Ozone']

# Convert 'Time' to numeric if it's in datetime format
X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds

# Initialize the XGBoost model
xg_reg = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1)

# Initialize KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=12)

# Prepare lists to store scores for each fold
mse_scores = []
mae_scores = []
r2_scores = []
mape_scores = []
nrmse_scores = []

# Perform cross-validation
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Fit the model on the training data
    xg_reg.fit(X_train, y_train)

    # Make predictions on the test data
    y_pred = xg_reg.predict(X_test)

    # Calculate evaluation metrics
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100  # MAPE calculation
    nrmse = np.sqrt(mse) / (y_test.max() - y_test.min()) * 100  # NRMSE calculation

    # Append scores for each fold
    mse_scores.append(mse)
    mae_scores.append(mae)
    r2_scores.append(r2)
    mape_scores.append(mape)
    nrmse_scores.append(nrmse)

# Calculate mean and standard deviation for each metric
mean_mse = np.mean(mse_scores)
std_mse = np.std(mse_scores)
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)
mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)
mean_mape = np.mean(mape_scores)
std_mape = np.std(mape_scores)
mean_nrmse = np.mean(nrmse_scores)
std_nrmse = np.std(nrmse_scores)

# Print evaluation metrics
print(f"Mean Squared Error: {round(mean_mse, 2)} ± {round(std_mse, 2)}")
print(f"Mean Absolute Error: {round(mean_mae, 2)} ± {round(std_mae, 2)}")
print(f"R^2 Score: {round(mean_r2, 2)} ± {round(std_r2, 2)}")
print(f"Mean Absolute Percentage Error: {round(mean_mape, 2)} ± {round(std_mape, 2)}%")
print(f"Normalized RMSE: {round(mean_nrmse, 2)} ± {round(std_nrmse, 2)}%")


C:\Users\Faculty\AppData\Local\Temp\ipykernel_35448\1334393487.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds


Mean Squared Error: 562.25 ± 3.7
Mean Absolute Error: 16.01 ± 0.04
R^2 Score: 0.51 ± 0.0
Mean Absolute Percentage Error: 225.88 ± 3.37%
Normalized RMSE: 11.86 ± 0.04%


MLR

In [6]:
# Load the dataset
all_available = pd.read_csv(r'D:\Manoj_Honors\RF_csv\all_available(t-1).csv')

# Prepare the feature set and target variable
X = all_available[['Time', 'Station_ID', 'NO2', 'Average_NO2_t-1']]
y = all_available[['PM2.5', 'Ozone']]

# Convert 'Time' to numeric if it's in datetime format
X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds

# Initialize the Multiple Linear Regression model
mlr = LinearRegression()

# Initialize KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=12)

# Prepare lists to store scores for each fold
mse_scores = []
mae_scores = []
r2_scores = []
mape_scores = []
nrmse_scores = []

# Perform cross-validation
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Fit the model on the training data
    mlr.fit(X_train, y_train)

    # Make predictions on the test data
    y_pred = mlr.predict(X_test)

    # Calculate evaluation metrics
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Calculate MAPE and NRMSE for each target variable
    mape_pm25 = np.mean(np.abs((y_test['PM2.5'] - y_pred[:, 0]) / y_test['PM2.5'])) * 100
    mape_ozone = np.mean(np.abs((y_test['Ozone'] - y_pred[:, 1]) / y_test['Ozone'])) * 100
    nrmse_pm25 = np.sqrt(mean_squared_error(y_test['PM2.5'], y_pred[:, 0])) / (y_test['PM2.5'].max() - y_test['PM2.5'].min()) * 100
    nrmse_ozone = np.sqrt(mean_squared_error(y_test['Ozone'], y_pred[:, 1])) / (y_test['Ozone'].max() - y_test['Ozone'].min()) * 100

    # Append metrics to lists
    mse_scores.append(mse)
    mae_scores.append(mae)
    r2_scores.append(r2)
    mape_scores.append((mape_pm25 + mape_ozone) / 2)
    nrmse_scores.append((nrmse_pm25 + nrmse_ozone) / 2)

# Calculate mean and standard deviation for each metric
mean_mse = np.mean(mse_scores)
std_mse = np.std(mse_scores)
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)
mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)
mean_mape = np.mean(mape_scores)
std_mape = np.std(mape_scores)
mean_nrmse = np.mean(nrmse_scores)
std_nrmse = np.std(nrmse_scores)

# Print evaluation metrics
print(f"Mean Squared Error: {round(mean_mse, 2)} ± {round(std_mse, 2)}")
print(f"Mean Absolute Error: {round(mean_mae, 2)} ± {round(std_mae, 2)}")
print(f"R^2 Score: {round(mean_r2, 2)} ± {round(std_r2, 2)}")
print(f"Mean Absolute Percentage Error: {round(mean_mape, 2)} ± {round(std_mape, 2)}%")
print(f"Normalized RMSE: {round(mean_nrmse, 2)} ± {round(std_nrmse, 2)}%")


C:\Users\Faculty\AppData\Local\Temp\ipykernel_262544\854710221.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds


Mean Squared Error: 4185.71 ± 32.7
Mean Absolute Error: 42.88 ± 0.09
R^2 Score: 0.1 ± 0.0
Mean Absolute Percentage Error: 278.21 ± 3.21%
Normalized RMSE: 12.33 ± 0.05%


SVR

In [7]:
# Load the dataset
all_available = pd.read_csv(r'C:\Users\Faculty\Desktop\Manoj_Honors\RF_csv\all_available.csv')

# Prepare the feature set and target variable
X = all_available[['Time', 'Station_ID', 'PM2.5', 'Ozone']]
y = all_available['NO2']

# Convert 'Time' to numeric if it's in datetime format
X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds

# Initialize the Support Vector Machine model
svm = SVR(kernel='rbf') 

# Initialize KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=12)

# Prepare lists to store scores for each fold
mse_scores = []
mae_scores = []
r2_scores = []

# Perform cross-validation
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Fit the model on the training data
    svm.fit(X_train, y_train)
    
    # Make predictions on the test data
    y_pred = svm.predict(X_test)
    
    # Calculate evaluation metrics
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    mse_scores.append(mse)
    mae_scores.append(mae)
    r2_scores.append(r2)

# Calculate mean and standard deviation for each metric
mean_mse = np.mean(mse_scores)
std_mse = np.std(mse_scores)
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)
mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)

# Print evaluation metrics
print(f"Mean Squared Error: {round(mean_mse, 2)} ± {round(std_mse, 2)}")
print(f"Mean Absolute Error: {round(mean_mae, 2)} ± {round(std_mae, 2)}")
print(f"R^2 Score: {round(mean_r2, 2)} ± {round(std_r2, 2)}")


C:\Users\Faculty\AppData\Local\Temp\ipykernel_15556\1845975102.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds


Mean Squared Error: 1345.21 ± 17.03
Mean Absolute Error: 22.42 ± 0.08
R^2 Score: -0.08 ± 0.0


In [2]:
# Load the dataset
all_available = pd.read_csv(r'D:\Manoj_Honors\RF_csv\all_available(t-1)_with_spatial.csv')

# Prepare the feature set with NO2 and the target variables
X = all_available[['Time', 'Lat', 'Lon', 'PM2.5','Average_PM2.5_t-1']]  # Use NO2 as the input feature
y = all_available[['NO2', 'Ozone']]  # Target variables for PM2.5 and Ozone

# Convert 'Time' to numeric if it's in datetime format
X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds

# Initialize the Random Forest model
rf = RandomForestRegressor(n_estimators=100, random_state=12)

# Initialize KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=12)

# Prepare lists to store scores for each fold for both targets
mse_scores_no2 = []
mae_scores_no2 = []
r2_scores_no2 = []
mape_scores_no2 = []
nrmse_scores_no2 = []

mse_scores_o3 = []
mae_scores_o3 = []
r2_scores_o3 = []
mape_scores_o3 = []
nrmse_scores_o3 = []

# Perform cross-validation
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Fit the model on the training data
    rf.fit(X_train, y_train)

    # Make predictions on the test data
    y_pred = rf.predict(X_test)

    # Split predictions back into NO2 and Ozone
    y_pred_no2 = y_pred[:, 0]
    y_pred_o3 = y_pred[:, 1]

    # Calculate evaluation metrics for NO2
    mse_no2 = mean_squared_error(y_test['NO2'], y_pred_no2)
    mae_no2 = mean_absolute_error(y_test['NO2'], y_pred_no2)
    r2_no2 = r2_score(y_test['NO2'], y_pred_no2)
    mape_no2 = np.mean(np.abs((y_test['NO2'] - y_pred_no2) / y_test['NO2'])) * 100
    nrmse_no2 = np.sqrt(mse_no2) / (y_test['NO2'].max() - y_test['NO2'].min())

    # Calculate evaluation metrics for Ozone
    mse_o3 = mean_squared_error(y_test['Ozone'], y_pred_o3)
    mae_o3 = mean_absolute_error(y_test['Ozone'], y_pred_o3)
    r2_o3 = r2_score(y_test['Ozone'], y_pred_o3)
    mape_o3 = np.mean(np.abs((y_test['Ozone'] - y_pred_o3) / y_test['Ozone'])) * 100
    nrmse_o3 = np.sqrt(mse_o3) / (y_test['Ozone'].max() - y_test['Ozone'].min())

    # Append metrics for NO2
    mse_scores_no2.append(mse_no2)
    mae_scores_no2.append(mae_no2)
    r2_scores_no2.append(r2_no2)
    mape_scores_no2.append(mape_no2)
    nrmse_scores_no2.append(nrmse_no2)

    # Append metrics for Ozone
    mse_scores_o3.append(mse_o3)
    mae_scores_o3.append(mae_o3)
    r2_scores_o3.append(r2_o3)
    mape_scores_o3.append(mape_o3)
    nrmse_scores_o3.append(nrmse_o3)

# Calculate mean and standard deviation for each metric for NO2
mean_mse_no2 = np.mean(mse_scores_no2)
std_mse_no2 = np.std(mse_scores_no2)
mean_mae_no2 = np.mean(mae_scores_no2)
std_mae_no2 = np.std(mae_scores_no2)
mean_r2_no2 = np.mean(r2_scores_no2)
std_r2_no2 = np.std(r2_scores_no2)
mean_mape_no2 = np.mean(mape_scores_no2)
std_mape_no2 = np.std(mape_scores_no2)
mean_nrmse_no2 = np.mean(nrmse_scores_no2)
std_nrmse_no2= np.std(nrmse_scores_no2)

# Calculate mean and standard deviation for each metric for Ozone
mean_mse_o3 = np.mean(mse_scores_o3)
std_mse_o3 = np.std(mse_scores_o3)
mean_mae_o3 = np.mean(mae_scores_o3)
std_mae_o3 = np.std(mae_scores_o3)
mean_r2_o3 = np.mean(r2_scores_o3)
std_r2_o3 = np.std(r2_scores_o3)
mean_mape_o3 = np.mean(mape_scores_o3)
std_mape_o3 = np.std(mape_scores_o3)
mean_nrmse_o3 = np.mean(nrmse_scores_o3)
std_nrmse_o3 = np.std(nrmse_scores_o3)

# Print evaluation metrics for NO2
print(f"NO2 - Mean Squared Error: {round(mean_mse_no2, 2)} ± {round(std_mse_no2, 2)}")
print(f"NO2 - Mean Absolute Error: {round(mean_mae_no2, 2)} ± {round(std_mae_no2, 2)}")
print(f"NO2 - R^2 Score: {round(mean_r2_no2, 2)} ± {round(std_r2_no2, 2)}")
print(f"NO2 - Mean Absolute Percentage Error: {round(mean_mape_no2, 2)}% ± {round(std_mape_no2, 2)}%")
print(f"NO2 - Normalized RMSE: {round(mean_nrmse_no2, 4)} ± {round(std_nrmse_no2, 4)}")

# Print evaluation metrics for Ozone
print(f"Ozone - Mean Squared Error: {round(mean_mse_o3, 2)} ± {round(std_mse_o3, 2)}")
print(f"Ozone - Mean Absolute Error: {round(mean_mae_o3, 2)} ± {round(std_mae_o3, 2)}")
print(f"Ozone - R^2 Score: {round(mean_r2_o3, 2)} ± {round(std_r2_o3, 2)}")
print(f"Ozone - Mean Absolute Percentage Error: {round(mean_mape_o3, 2)}% ± {round(std_mape_o3, 2)}%")
print(f"Ozone - Normalized RMSE: {round(mean_nrmse_o3, 4)} ± {round(std_nrmse_o3, 4)}")
import joblib
model_output_path = r'D:\Manoj_Honors\random_forest_model_NO2_Ozone.joblib'
joblib.dump(rf, model_output_path)

print(f"Model saved to {model_output_path}")


C:\Users\Faculty\AppData\Local\Temp\ipykernel_8348\3498463340.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds


NO2 - Mean Squared Error: 151.17 ± 2.87
NO2 - Mean Absolute Error: 6.03 ± 0.02
NO2 - R^2 Score: 0.88 ± 0.0
NO2 - Mean Absolute Percentage Error: 35.06% ± 1.6%
NO2 - Normalized RMSE: 0.0246 ± 0.0002
Ozone - Mean Squared Error: 198.46 ± 3.13
Ozone - Mean Absolute Error: 7.52 ± 0.05
Ozone - R^2 Score: 0.83 ± 0.0
Ozone - Mean Absolute Percentage Error: 99.09% ± 2.58%
Ozone - Normalized RMSE: 0.0704 ± 0.0006
Model saved to D:\Manoj_Honors\random_forest_model_NO2_Ozone.joblib


In [2]:
import pandas as pd
import joblib

# File paths
model_path = r"D:\Manoj_Honors\random_forest_model_NO2_Ozone.joblib"  # Assuming one model for both PM2.5 and NO2
input_csv_path = r"D:\Manoj_Honors\Imputed_Predictions(Univariate,(O3)).csv"
output_csv_path = r"D:\Manoj_Honors\Imputed_Predictions(Univariate,(O3,PM2.5)).csv"

# Load the model
with open(model_path, 'rb') as model_file:
    rf_model = joblib.load(model_file)

# Load the input data
df = pd.read_csv(input_csv_path)

# Check the required columns
required_columns = ['Time', 'Lat', 'Lon', 'NO2', 'PM2.5', 'Average_PM2.5_t-1', 'Ozone']
for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# Identify rows where both PM2.5 and NO2 are missing and Ozone is available
mask_missing_pm25_and_no2 = df['Ozone'].isna() & df['NO2'].isna() & df['PM2.5'].notna()

# Count rows with missing PM2.5 and NO2
missing_count = mask_missing_pm25_and_no2.sum()
print(f"Number of rows with missing PM2.5 and NO2 (but Ozone available): {missing_count}")

# Filter the rows where both PM2.5 and NO2 are missing and Ozone is available
rows_to_impute = df[mask_missing_pm25_and_no2]

# Prepare the features for imputation
X_to_impute = rows_to_impute[['Time', 'Lat', 'Lon', 'PM2.5','Average_PM2.5_t-1']]

# Convert 'Time' to seconds since epoch (as the model was likely trained on this format)
X_to_impute['Time'] = pd.to_datetime(X_to_impute['Time'], errors='coerce').astype('int64') // 10**9

# Ensure the columns are in the same order as during model training
train_columns = ['Time', 'Lat', 'Lon', 'PM2.5','Average_PM2.5_t-1']
X_to_impute = X_to_impute[train_columns]

# Impute the missing values using the model
imputed_values = rf_model.predict(X_to_impute)

# Assign the imputed values back to the appropriate columns
# Assume the model returns imputed values for both PM2.5 and NO2
df.loc[mask_missing_pm25_and_no2, ['Ozone', 'NO2']] = imputed_values.reshape(-1, 2)

# Save the updated dataframe with imputed values to a new CSV
df.to_csv(output_csv_path, index=False)

print(f"Imputed rows and saved the updated data to {output_csv_path}")


Number of rows with missing PM2.5 and NO2 (but Ozone available): 3406


C:\Users\Faculty\AppData\Local\Temp\ipykernel_17492\160658304.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_to_impute['Time'] = pd.to_datetime(X_to_impute['Time'], errors='coerce').astype('int64') // 10**9


Imputed rows and saved the updated data to D:\Manoj_Honors\Imputed_Predictions(Univariate,(O3,PM2.5)).csv


In [5]:
import joblib

# Save the model
joblib.dump(rf, 'random_forest_model(NO2,O3).pkl')

print("Model saved successfully!")


Model saved successfully!


In [2]:
# Load the dataset
all_available = pd.read_csv('D:\Manoj_Honors\Final\All_available(Temporal).csv')

all_available = all_available.dropna()
print(len(all_available))

X = all_available[['Time', 'Lat','Lon','Ozone_prev_day','Ozone_prev_week','Spatial_Avg_Ozone','Ozone']]
y = all_available['NO2']

# Convert 'Time' to numeric if it's in datetime format
X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds


# Initialize the Random Forest model
rf = RandomForestRegressor(n_estimators=100, random_state=12)

# Initialize KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=12)

# Prepare lists to store scores for each fold
mse_scores = []
mae_scores = []
r2_scores = []
mape_scores = []
nrmse_scores = []

# Perform cross-validation
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Fit the model on the training data
    rf.fit(X_train, y_train)

    # Make predictions on the test data
    y_pred = rf.predict(X_test)

    # Calculate evaluation metrics
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    nrmse = np.sqrt(mse) / (y_test.max() - y_test.min())

    mse_scores.append(mse)
    mae_scores.append(mae)
    r2_scores.append(r2)
    mape_scores.append(mape)
    nrmse_scores.append(nrmse)

# Calculate mean and standard deviation for each metric
mean_mse = np.mean(mse_scores)
std_mse = np.std(mse_scores)
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)
mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)
mean_mape = np.mean(mape_scores)
std_mape = np.std(mape_scores)
mean_nrmse = np.mean(nrmse_scores)
std_nrmse = np.std(nrmse_scores)

# Print evaluation metrics
print(f"Mean Squared Error: {round(mean_mse, 2)} ± {round(std_mse, 2)}")
print(f"Mean Absolute Error: {round(mean_mae, 2)} ± {round(std_mae, 2)}")
print(f"R^2 Score: {round(mean_r2, 2)} ± {round(std_r2, 2)}")
print(f"Mean Absolute Percentage Error: {round(mean_mape, 2)}% ± {round(std_mape, 2)}%")
print(f"Normalized RMSE: {round(mean_nrmse, 4)} ± {round(std_nrmse, 4)}")

# import pickle

# with open('random_forest_model_PM2.5.pkl', 'wb') as model_file:
#     pickle.dump(rf, model_file)


<>:2: SyntaxWarning: invalid escape sequence '\M'
<>:2: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Faculty\AppData\Local\Temp\ipykernel_11504\653445196.py:2: SyntaxWarning: invalid escape sequence '\M'
  all_available = pd.read_csv('D:\Manoj_Honors\Final\All_available(Temporal).csv')


764702


C:\Users\Faculty\AppData\Local\Temp\ipykernel_11504\653445196.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds


Mean Squared Error: 159.93 ± 1.79
Mean Absolute Error: 6.6 ± 0.02
R^2 Score: 0.87 ± 0.0
Mean Absolute Percentage Error: 36.18% ± 1.28%
Normalized RMSE: 0.0254 ± 0.0001
